# Task 4: Zero-Shot Classification (Multi-Label)

Classifies posts into one or more of 6 categories using Claude API (multi-label binary
indicator columns).

**Scope decision (confirmed with project owner):**
- **Stratified sample**, not the full ~126,400-row dataset: sourdough, banana_bread, and
  dalgona_coffee capped at 3,000 posts each (random sample, fixed seed); baked_oats (121) and
  feta_pasta (302) kept at full size since they're already small.
- **Model:** `claude-sonnet-5`.
- **Call mode:** Anthropic's async **Message Batches API** (50% cheaper than real-time calls,
  no manual rate-limit pacing needed).

**Important downstream consequence:** because this is a sample rather than the full dataset,
the file this notebook saves (`trends_combined_english_features.csv`, overwritten) will contain
**only the ~9,400 sampled/classified rows**, not all 126,410. Task 5's regression will therefore
run on this same stratified sample, not the full English-only dataset. This is a deliberate
consequence of the sampling decision, flagged here rather than applied silently.

In [1]:
import pandas as pd
import numpy as np
import json
import time
import os
import anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5"
LABELS = [
    "recipe_instructional", "personal_lifestyle", "media_repost",
    "meme_joke", "spam_low_content",
]

## Step 0: Load Task 3's output

In [2]:
combined = pd.read_csv("../../output/cleaned_data/trends_combined_english_features.csv")
print(combined.shape)

(126410, 32)


## Build the stratified sample

Cap sourdough, banana_bread, dalgona_coffee at 3,000 posts each (random, fixed seed for
reproducibility); keep baked_oats and feta_pasta at full size.

In [3]:
CAP = 3000
SEED = 42

sampled_parts = []
for trend, group in combined.groupby("trend"):
    n = min(len(group), CAP)
    sampled_parts.append(group.sample(n=n, random_state=SEED))

combined = pd.concat(sampled_parts, ignore_index=True)
print(combined.shape)
print(combined["trend"].value_counts())

# post_id used for classification is the row index in this sampled, reset-index frame
# (the source `id` column has ~1,098 duplicates dataset-wide, so it is not safe as a key)
combined = combined.reset_index(drop=True)
combined["post_id"] = combined.index

(9423, 32)
trend
banana_bread      3000
dalgona_coffee    3000
sourdough         3000
feta_pasta         302
baked_oats         121
Name: count, dtype: int64


## Step 1: Design the classification prompt/schema, test on a small batch first

Label definitions (5 labels; `trend_commentary` removed entirely, `spam_low_content` redefined
to require minimal substantive content specifically). Prompt finalized after two revision rounds
against real sampled post text (v1 conflated "promotional" with "spam"; v2 exposed the model
treating a hashtag tail as a spam proxy instead of judging caption substance on its own -- v3
below adds explicit clarifications to fix that):

- **recipe_instructional**: post provides or centers on a recipe/cooking instructions
  (ingredients, steps, "how to make X")
- **personal_lifestyle**: personal/narrative content about the poster's day, life, family, or
  mood, not primarily about the recipe/food itself
- **media_repost**: clearly reposted/shared content from another source (press release, news
  article, another account's or business's content, ad copy) rather than original personal
  content
- **meme_joke**: humor/meme/joke-framed content
- **spam_low_content**: post has minimal substantive content, e.g. just a hashtag string with no
  real caption, or generic engagement-bait text with little real connection to the food/recipe
  itself

Multi-label, not exhaustive: a post can receive 0, 1, or multiple labels. Zero labels is expected
and correct for posts that don't clearly fit any category -- do not force a label onto every
post. Labels are evaluated independently (one label's presence should not suppress another that
also applies), hashtags are ignored when judging substance, and promotional/commercial intent
alone is not a spam signal.

In [4]:
SYSTEM_PROMPT = """You are classifying Instagram/Facebook posts about COVID-era food trends (sourdough, banana bread, dalgona coffee, baked oats, feta pasta) into content-type categories.

For EACH post, assign zero, one, or multiple of the following labels. These are independent binary categories, not mutually exclusive, and not exhaustive: a post that doesn't clearly fit any of them should simply get zero labels, do not force a fit.

- recipe_instructional: post provides or centers on a recipe/cooking instructions (ingredients,   steps, "how to make X")
- personal_lifestyle: personal/narrative content about the poster's day, life, family, or mood,   not primarily about the recipe/food itself
- media_repost: clearly reposted/shared content from another source (press release, news article,   another account's or business's content, ad copy) rather than original personal content
- meme_joke: humor/meme/joke-framed content
- spam_low_content: post has minimal substantive content, e.g. just a hashtag string with no real   caption, or generic engagement-bait text with little real connection to the food/recipe itself

Important clarifications:
- Labels are independent. Evaluate each one on its own merits. A post can be both   recipe_instructional (if the caption has real recipe content) AND spam_low_content (if it is   also mostly hashtag padding) at the same time. Do not let one label's presence suppress another   that also applies.
- A real, substantive caption followed by hashtags is NOT spam_low_content just because hashtags   are present. Hashtags are standard Instagram convention and should be ignored when judging   substance. Only the caption content itself (ignoring the hashtag block) determines this label.
- Promotional or commercial intent alone does NOT make a post spam_low_content. A business account   taking orders, describing a product, or advertising a delivery service with a real, on-topic   caption is not spam just because it is commercial, judge only whether the caption itself is   substantive. If it is substantive but doesn't fit any other category, it should receive no   label, not spam_low_content.
- A post can legitimately receive zero labels. This is expected and correct, do not stretch a   definition to force a label onto every post.

Respond only with structured JSON, no other text."""

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "classifications": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "post_id": {"type": "integer"},
                    "labels": {
                        "type": "array",
                        "items": {"type": "string", "enum": LABELS},
                    },
                },
                "required": ["post_id", "labels"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["classifications"],
    "additionalProperties": False,
}

def build_user_content(posts_df):
    """posts_df: DataFrame with post_id and text columns."""
    payload = [{"post_id": int(r.post_id), "text": str(r.text)[:2000]} for r in posts_df.itertuples()]
    return "Classify these posts:\n\n" + json.dumps(payload, ensure_ascii=False)

In [5]:
# Test batch: 20 posts, direct (non-batch) call, manually inspect results against real text
test_posts = combined.sample(20, random_state=SEED)[["post_id", "text"]]

test_response = client.messages.create(
    model=MODEL,
    max_tokens=4000,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": build_user_content(test_posts)}],
    output_config={"format": {"type": "json_schema", "schema": LABEL_SCHEMA}},
)

test_text = next(b.text for b in test_response.content if b.type == "text")
test_result = json.loads(test_text)

test_lookup = {c["post_id"]: c["labels"] for c in test_result["classifications"]}
for r in test_posts.itertuples():
    labels = test_lookup.get(r.post_id, ["MISSING"])
    print(f"[{labels}]")
    print(f"  {str(r.text)[:180]}")
    print()

[['recipe_instructional']]
  You call it Dalgona, I call it iced coffee- my daily coffee uses erythritol or xylitol, occasionally I use brown sugar... 
2 tsp espresso powder 
(Add collagen in here at this stag

[[]]
  Today is the day! 🥤

After a long wait, we are ready to serve you the creamiest vegan plant-based Dalgona and Boba in town!

Are you #DalgonaSquad or #BobaGang? Tell us in the comm

[['personal_lifestyle', 'meme_joke']]
  Happy Sunday folks! Let’s play a little game, shall we? Leave the appropriate emoji/s in the comments below if since the start of quarantine, you’ve:⁣
⁣
☕️ Made whipped/dalgona/tik

[['personal_lifestyle']]
  Sometimes I need an extra special pick me up - nothing works better than a cup of @nescafeindia Dalgona Coffee to fuel the endless multitasking and kiddo management!

One of my fav

[['media_repost']]
  Good morning all~ start your day with our busanz (champaca) ft. cignature making dalgona coffee☕️ Did you guys try it yet?☺️
⋯⋯⋯⋯⋯⋯⋯
✧；credit to aut

**Manual check:** review the 20 test assignments above against the printed post text before
scaling up. If labels look reasonable, proceed to the full batch below.

## Step 2: Run classification on the full stratified sample via the Batch API

Batches of 15 posts per request. The batch ID is checkpointed to disk immediately after
submission — if the kernel restarts, re-running the next cell resumes polling the existing
batch instead of resubmitting.

In [5]:
BATCH_SIZE = 15
CHECKPOINT_PATH = "task4_batch_checkpoint.json"

def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

post_id_batches = list(chunk(combined["post_id"].tolist(), BATCH_SIZE))
print(f"{len(post_id_batches)} batches of up to {BATCH_SIZE} posts each, "
      f"{len(combined)} posts total")

629 batches of up to 15 posts each, 9423 posts total


In [6]:
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    batch_id = checkpoint["batch_id"]
    print(f"Resuming existing batch: {batch_id}")
else:
    id_to_text = dict(zip(combined["post_id"], combined["text"]))
    requests = []
    for i, id_batch in enumerate(post_id_batches):
        posts_df = pd.DataFrame({"post_id": id_batch, "text": [id_to_text[pid] for pid in id_batch]})
        requests.append(Request(
            custom_id=f"batch-{i}",
            params=MessageCreateParamsNonStreaming(
                model=MODEL,
                max_tokens=4000,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": build_user_content(posts_df)}],
                output_config={"format": {"type": "json_schema", "schema": LABEL_SCHEMA}},
            ),
        ))

    message_batch = client.messages.batches.create(requests=requests)
    batch_id = message_batch.id
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({"batch_id": batch_id, "n_requests": len(requests)}, f)
    print(f"Created batch: {batch_id} ({len(requests)} requests)")

Created batch: msgbatch_013Y1uBNpVTmhSu2gQVdMwSx (629 requests)


## Poll for completion

In [7]:
while True:
    batch = client.messages.batches.retrieve(batch_id)
    if batch.processing_status == "ended":
        break
    print(f"Status: {batch.processing_status}, counts: {batch.request_counts}")
    time.sleep(30)

print("Batch complete!")
print(f"Succeeded: {batch.request_counts.succeeded}")
print(f"Errored: {batch.request_counts.errored}")
print(f"Canceled: {batch.request_counts.canceled}")
print(f"Expired: {batch.request_counts.expired}")

Status: in_progress, counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=629, succeeded=0)


Status: in_progress, counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=629, succeeded=0)


Status: in_progress, counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=629, succeeded=0)


Status: in_progress, counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=629, succeeded=0)


Status: in_progress, counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=629, succeeded=0)


Status: in_progress, counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=629, succeeded=0)


Status: in_progress, counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=629, succeeded=0)


Batch complete!
Succeeded: 629
Errored: 0
Canceled: 0
Expired: 0


## Step 3: Parse results, add the 6 binary indicator columns

In [8]:
post_labels = {}  # post_id -> list of assigned labels
errored_custom_ids = []

for result in client.messages.batches.results(batch_id):
    if result.result.type == "succeeded":
        msg = result.result.message
        text = next(b.text for b in msg.content if b.type == "text")
        parsed = json.loads(text)
        for c in parsed["classifications"]:
            post_labels[c["post_id"]] = c["labels"]
    else:
        errored_custom_ids.append((result.custom_id, result.result.type))

print(f"Parsed labels for {len(post_labels)} / {len(combined)} posts")
print(f"Errored/non-succeeded requests: {len(errored_custom_ids)}")
if errored_custom_ids:
    print(errored_custom_ids[:10])

Parsed labels for 9423 / 9423 posts
Errored/non-succeeded requests: 0


In [9]:
for label in LABELS:
    combined[f"is_{label}"] = 0

missing_post_ids = []
for pid, labels in post_labels.items():
    for label in labels:
        combined.loc[combined["post_id"] == pid, f"is_{label}"] = 1

for pid in combined["post_id"]:
    if pid not in post_labels:
        missing_post_ids.append(pid)

print(f"Posts with no classification result (left all-0): {len(missing_post_ids)}")

Posts with no classification result (left all-0): 0


## Step 4: Sanity-check the classification output

In [10]:
label_cols = [f"is_{label}" for label in LABELS]
print("Label prevalence (count and % of sampled posts):")
for col in label_cols:
    count = combined[col].sum()
    pct = combined[col].mean() * 100
    print(f"  {col}: {count} ({pct:.1f}%)")

print()
print("Posts with 0 labels assigned:", (combined[label_cols].sum(axis=1) == 0).sum())

Label prevalence (count and % of sampled posts):
  is_recipe_instructional: 2430 (25.8%)
  is_personal_lifestyle: 3074 (32.6%)
  is_media_repost: 1622 (17.2%)
  is_meme_joke: 566 (6.0%)
  is_spam_low_content: 1967 (20.9%)

Posts with 0 labels assigned: 1440


In [11]:
# Manual spot-check: a few random posts per label
for label in LABELS:
    col = f"is_{label}"
    subset = combined[combined[col] == 1]
    if len(subset) == 0:
        print(f"--- {label}: no posts assigned ---\n")
        continue
    sample_n = min(3, len(subset))
    print(f"--- {label} ({subset.shape[0]} posts) ---")
    for r in subset.sample(sample_n, random_state=SEED).itertuples():
        print(f"  {str(r.text)[:150]}")
    print()

--- recipe_instructional (2430 posts) ---
  GLUTEN FREE + Vegan | Chocolate Chip 🍌 Banana Bread | SUPER MOIST | 1 BOWL | 9 Ingredients
⠀⠀⠀⠀⠀⠀⠀⠀⠀
Celebrate Father’s Day by baking this super easy,
  Have you ever made homemade butter? I'm not sure there's anything more luxurious than spreading homemade butter onto a warm slice of sourdough bread. 
  It’s Friday & we made it to May 🥳so let’s play a game called “how many banana🍌things can you bake👩🏼‍🍳during🧁quarantine?” Just a handful of ingredients

--- personal_lifestyle (3074 posts) ---
  Last night's simple (no cooking!) but indulgent supper. Sourdough that came out of the oven in the morning, @columbuscraftmeats dry sausage (the unwra
  the sunday roast is perhaps my favourite british culinary tradition: to an american like me, it’s a comforting reminder of a thanksgiving meal, minus 
  Same dough as previous post but shaped differently.
Our favourite sweet loaf for today’s breakfast date🍞100% naturally leavened fruity sourdough brioc

## Step 5: Multicollinearity check between labels (co-occurrence correlation matrix)

In [12]:
corr_matrix = combined[label_cols].corr()
print("Label co-occurrence correlation matrix:")
print(corr_matrix.round(3))

Label co-occurrence correlation matrix:
                         is_recipe_instructional  is_personal_lifestyle  \
is_recipe_instructional                    1.000                 -0.051   
is_personal_lifestyle                     -0.051                  1.000   
is_media_repost                           -0.096                 -0.264   
is_meme_joke                              -0.108                 -0.007   
is_spam_low_content                       -0.293                 -0.336   

                         is_media_repost  is_meme_joke  is_spam_low_content  
is_recipe_instructional           -0.096        -0.108               -0.293  
is_personal_lifestyle             -0.264        -0.007               -0.336  
is_media_repost                    1.000        -0.057               -0.065  
is_meme_joke                      -0.057         1.000               -0.032  
is_spam_low_content               -0.065        -0.032                1.000  


Report only, not a blocker for Task 5 — VIF will be checked again across the *full* predictor
set (rule-based + zero-shot + trend dummies) in Task 5 Step 7.

## Step 6: Save the updated file

Overwrites `trends_combined_english_features.csv` (Task 3's output) with the zero-shot label
columns added, **scoped to the ~9,400-row stratified sample** (see the scope note at the top of
this notebook — this is now the working dataset for Task 5's regression, not the full 126,410
rows).

In [13]:
combined = combined.drop(columns=["post_id"])
combined.to_csv("../../output/cleaned_data/trends_combined_english_features.csv", index=False)
print(f"Saved trends_combined_english_features.csv, shape: {combined.shape}")
print(combined.columns.tolist())

Saved trends_combined_english_features.csv, shape: (9423, 37)
['content_type', 'creation_time', 'hashtags', 'id', 'is_branded_content', 'lang', 'match_type', 'mcl_url', 'modified_time', 'multimedia', 'post_owner.id', 'post_owner.name', 'post_owner.type', 'post_owner.username', 'statistics.comment_count', 'statistics.like_count', 'statistics.views', 'statistics.views_date_last_refreshed', 'text', 'date_parsed', 'month', 'is_covid_framed', 'trend', 'word_count', 'log_word_count', 'sentiment_score', 'hashtag_count', 'exclamation_count', 'question_count', 'emoji_count', 'log_likes', 'log_comments', 'is_recipe_instructional', 'is_personal_lifestyle', 'is_media_repost', 'is_meme_joke', 'is_spam_low_content']
